# 11 - Phase 2 follow-up experiments (baselines, validation sweeps, per-stage timing)

Closes the gaps listed in the Phase 2 self-assessment, with the evaluated weights and the same split:

| Part | What it measures | Split | Est. time (T4) |
|---|---|---|---|
| A | Detector baseline: COCO-pretrained YOLO11s **as is** (no fine-tuning), sweep on validation, test once | val, test | 5 min |
| B | Segmentation baselines: COCO-pretrained YOLO11s-seg as is (val, test); SAM 2.1-B oracle box on 300 **validation** images | val, test | 8 min |
| C | Fine-tuned segmenter: mask confidence swept on validation; test once at the chosen value | val, test | 8 min |
| D | Full pipeline on validation: mask-match IoU and minimum mask share swept; per-stage p50/p95 on the T4 | val | 7 min |
| E | ONNX detector: mAP on the test split vs PyTorch; box/score channel diagnosis (A9b unchanged) | test | 6 min |
| F | Second detector recipe: YOLO11n, same settings (optional, `RUN_EXTRA_DETECTOR`) | train, val, test | about 40 min |

Rules: every choice is made on validation with a rule stated before the run; the test split is evaluated once per setting; no threshold of the served manifest is changed by this notebook; no team photo is used. Results are saved after each part to `OUTPUT_BASE/phase2_gap_experiments_<time>/` so a disconnect loses at most one part.

**Runtime:** GPU (T4). Run all cells in order from a new runtime. The W&B key is read from Colab Secrets (`WANDB_API_KEY`) or a hidden prompt.


In [ ]:
# 1. Configuration
DRIVE_PROJECT = "/content/drive/MyDrive/capstone /smart-pet-vision"   # runs/manifest.json, weights, runs/data/metadata.csv
OUTPUT_BASE = "/content/drive/MyDrive/capstone /Submission/FINAL PHASE_2/03_Evidence"
REPO_URL, REPO_REV = "https://github.com/Yasser-20-alhajlah/smart-pet-vision", "phase2-gap-closure"   # "main" once PR #5 is merged
RUN_EXTRA_DETECTOR = True      # Part F, about 40 minutes on a T4
LOG_TO_WANDB = True
WANDB_ENTITY, WANDB_PROJECT = "yasser-alhajlah-kaust", "smart-pet-vision"


In [ ]:
# 2. Setup: code at a pinned commit, pinned packages, data rebuilt from the saved split
import os, sys, json, time, shutil, subprocess, platform
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
WORK = Path("/content/spv11")
if WORK.exists(): raise RuntimeError("Use a new runtime: /content/spv11 already exists")
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORK / "repo")], check=True)
subprocess.run(["git", "-C", str(WORK / "repo"), "checkout", "--quiet", REPO_REV], check=True)
SHA = subprocess.run(["git", "-C", str(WORK / "repo"), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
REPO = WORK / "repo" / "smart-pet-vision" / "phase2"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO.parent / "requirements.txt")], check=True)
sys.path[:0] = [str(REPO / "src"), str(REPO / "tools")]
import torch, numpy as np, pandas as pd
assert torch.cuda.is_available(), "Select a GPU runtime (T4)"
from spv.common import load_config, save_json
from spv import data, detect, segment
import phase2_gap_experiments as gx, preflight_release as pf

DP = Path(DRIVE_PROJECT)
MAN = DP / "runs" / "manifest.json"
PRE = pf.run(MAN)                                   # files, weight SHA-256, operating point, review_mode
META = DP / "runs" / "data" / "metadata.csv"
if not META.is_file(): raise FileNotFoundError(f"Saved split not found: {META}")
cfg = load_config(REPO / "configs" / "config.yaml")
cfg["paths"].update(raw=str(WORK / "data/raw"), processed=str(WORK / "data/processed"), runs=str(WORK / "runs"))
(WORK / "runs/data").mkdir(parents=True); shutil.copy2(META, WORK / "runs/data/metadata.csv")
df = data.ensure_data(cfg)                          # downloads Oxford-IIIT Pet, rebuilds masks and YOLO folders (about 10 min)
assert len(df) == 7335 and df.split.value_counts().to_dict() == {"train": 5123, "val": 1106, "test": 1106}, df.split.value_counts()
IMG, PROC = data.dataset_dir(cfg) / "images", Path(cfg["paths"]["processed"])
VAL, TEST = df[df.split == "val"], df[df.split == "test"]
from spv.pipeline import Manifest
M = Manifest.load(MAN)
OUT = Path(OUTPUT_BASE) / f"phase2_gap_experiments_{STAMP}"; OUT.mkdir(parents=True)
RESULTS = {"source_commit": SHA, "weights_sha256": PRE["weights_sha256"], "device": torch.cuda.get_device_name(0),
           "versions": {"torch": torch.__version__, "python": platform.python_version()}, "split": "Oxford-IIIT Pet, saved metadata.csv"}
def save(part, obj):
    RESULTS[part] = obj; save_json(RESULTS, OUT / "results.json"); print(f"saved {part} -> {OUT/'results.json'}")
print("commit", SHA, "| output", OUT)


In [ ]:
# A. Detector baseline: COCO-pretrained YOLO11s used as is (cat=15, dog=16 mapped to ours)
from ultralytics import YOLO
coco_names = YOLO("yolo11s.pt").names
DET_COCO = gx.remap_yolo_dataset(PROC / "yolo_detect", WORK / "yolo_detect_coco", coco_names)
val_gts = detect.gt_for(VAL)
preds = gx.predict_mapped("yolo11s.pt", [IMG / f"{n}.jpg" for n in val_gts], conf=0.05, iou=0.5, imgsz=640, class_map=gx.COCO_TO_OURS)
sweep = gx.sweep_from_predictions(preds, val_gts, cfg["detection"]["conf_grid"], 0.5)
op = detect.choose_operating_point(sweep, min_recall=0.80)     # same rule as the fine-tuned detector
test_gts = detect.gt_for(TEST)
tpreds = gx.predict_mapped("yolo11s.pt", [IMG / f"{n}.jpg" for n in test_gts], conf=0.05, iou=0.5, imgsz=640, class_map=gx.COCO_TO_OURS)
from spv.detect import score
save("A_detector_coco_baseline", {
    "model": "yolo11s.pt (COCO, not fine-tuned)", "imgsz": 640,
    "val": gx.ultralytics_val("yolo11s.pt", DET_COCO, "val", 640, classes=[15, 16]),
    "val_sweep_nms0.5": sweep.to_dict("records"), "operating_point": op,
    "test": gx.ultralytics_val("yolo11s.pt", DET_COCO, "test", 640, classes=[15, 16]),
    "test_at_operating_point": score(tpreds, test_gts, op["conf"])})


In [ ]:
# B. Segmentation baselines: COCO-pretrained YOLO11s-seg as is; SAM 2.1-B oracle box on validation
SEG_COCO = gx.remap_yolo_dataset(PROC / "yolo_segment", WORK / "yolo_segment_coco", YOLO("yolo11s-seg.pt").names)
coco_seg = {}
for split, part in (("val", VAL), ("test", TEST)):
    sw = gx.seg_conf_sweep("yolo11s-seg.pt", part, IMG, PROC, [0.25], 640, classes=[15, 16]).iloc[0].to_dict()
    coco_seg[split] = {**gx.ultralytics_val("yolo11s-seg.pt", SEG_COCO, split, 640, classes=[15, 16]), **sw}
val300 = VAL.sample(300, random_state=42)
sam_sum, _ = segment.evaluate(segment.sam_masks(val300, IMG), val300, PROC)
save("B_segmentation_baselines", {"yolo11s_seg_coco_as_is_conf0.25": coco_seg,
                                  "sam2.1_b_oracle_box_val300": sam_sum, "sam_val_sample_seed": 42})


In [ ]:
# C. Fine-tuned segmenter: mask confidence chosen on validation (rule: highest val mIoU; ties -> higher conf)
confs = [0.10, 0.25, 0.40, 0.55]
seg_sweep = gx.seg_conf_sweep(M.segmenter, VAL, IMG, PROC, confs, 640)
chosen = gx.choose_seg_conf(seg_sweep)
test_at_chosen = gx.seg_conf_sweep(M.segmenter, TEST, IMG, PROC, [chosen], 640).iloc[0].to_dict()
save("C_segmenter_conf_on_validation", {"val_sweep": seg_sweep.to_dict("records"), "rule": "highest val mIoU, ties to higher conf",
                                        "chosen_conf": chosen, "served_conf": M.segmenter_conf, "test_at_chosen": test_at_chosen,
                                        "val_mask_mAP": gx.ultralytics_val(M.segmenter, PROC / "yolo_segment" / "data.yaml", "val", 640)})


In [ ]:
# D. Full pipeline on validation (T4): mask-match IoU and minimum mask share; per-stage percentiles
# Rules stated before the run: largest match IoU whose false 'segmentation_unmatched' share on val <= 1%;
# largest minimum mask share whose false 'too small' share on val <= 1% (every val image holds one large animal).
paths = [IMG / f"{n}.jpg" for n in VAL.image]
sweep_rows, runs = [], {}
for miou in [0.05, 0.10, 0.25, 0.50]:
    rec = gx.run_pipeline(MAN, paths, overrides={"mask_match_iou": miou}, warmup=10)
    runs[miou] = rec
    s = gx.referral_summary(rec)
    sweep_rows.append({"mask_match_iou": miou, "unmatched": s["unmatched"], "unmatched_share": s["unmatched"] / s["n"],
                       "referred": s["referred"], "referred_share": s["referred"] / s["n"]})
match_sweep = pd.DataFrame(sweep_rows)
base = runs[0.10]
frac_sweep = gx.min_mask_fraction_sweep(base, [0.01, 0.02, 0.05, 0.10, 0.15])
stages = gx.stage_percentiles(base)
base.to_csv(OUT / "pipeline_val_records.csv", index=False)
save("D_pipeline_validation", {"mask_match_sweep": match_sweep.to_dict("records"),
    "chosen_mask_match_iou": gx.largest_within(match_sweep, "mask_match_iou", "unmatched_share", 0.01),
    "min_mask_fraction_sweep": frac_sweep.to_dict("records"),
    "chosen_min_mask_fraction": gx.largest_within(frac_sweep, "min_mask_fraction", "share", 0.01),
    "served": {"mask_match_iou": M.mask_match_iou, "min_mask_fraction": M.min_mask_fraction},
    "stage_percentiles_T4": stages.to_dict("records"), "referrals_at_served_settings": gx.referral_summary(base),
    "conditions": "Tesla T4, PyTorch, batch 1, 1,106 validation images, 10 warm-up, in-process"})
print(match_sweep); print(frac_sweep); print(stages)


In [ ]:
# E. ONNX detector: same metric on the test split; box/score channel diagnosis. A9b (<= 0.001) is not changed.
import tempfile, onnx_channel_diagnosis as diag
tmp = Path(tempfile.mkdtemp()); src = tmp / "best.pt"; shutil.copy2(M.detector, src)
onnx_path = Path(YOLO(str(src)).export(format="onnx", imgsz=640, opset=17, simplify=False, dynamic=False))
DET_YAML = PROC / "yolo_detect" / "data.yaml"
save("E_onnx", {"pytorch_test": gx.ultralytics_val(M.detector, DET_YAML, "test", 640),
                "onnx_test": gx.ultralytics_val(onnx_path, DET_YAML, "test", 640),
                "channel_diagnosis": {k: v for k, v in diag.run(src, onnx_path, imgsz=640, n=4, seed=0).items() if k != "per_input"}})
# E2. Hypothesis: the gap comes from the PyTorch convolution backend, not the export. Same protocol, reported whatever the result.
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]); name = sess.get_inputs()[0].name
first = lambda r: r[0] if isinstance(r, (list, tuple)) else r
backend = {}
for tag, mkldnn, threads in (("default", True, None), ("mkldnn_off", False, None), ("mkldnn_off_1thread", False, 1)):
    torch.backends.mkldnn.enabled = mkldnn
    if threads: torch.set_num_threads(threads)
    net = YOLO(str(src)).model.float().cpu().eval(); torch.manual_seed(0)
    diffs = []
    for _ in range(4):
        x = torch.rand(1, 3, 640, 640)
        with torch.no_grad(): ref = first(net(x)).numpy()
        diffs.append(float(np.abs(sess.run(None, {name: x.numpy()})[0] - ref).max()))
    backend[tag] = max(diffs)
torch.backends.mkldnn.enabled = True
save("E2_onnx_pytorch_backend", {"max_abs_diff_by_pytorch_backend": backend, "tolerance": 0.001,
                                 "note": "Diagnostic only; A9b stays as recorded (0.0018) and is not re-scored."})


In [ ]:
# F. (optional) Second detector recipe: YOLO11n, same settings as the selected YOLO11s run
if RUN_EXTRA_DETECTOR:
    import copy; cfg_n = copy.deepcopy(cfg); cfg_n["detection"]["model"] = "yolo11n.pt"
    w_n = detect.train(cfg_n, DET_YAML, name="yolo11n_det")
    sweep_n = detect.threshold_sweep(cfg_n, w_n, VAL, IMG)
    op_n = detect.choose_operating_point(sweep_n, min_recall=0.80)
    tp_n = detect.predict_all(w_n, [IMG / f"{n}.jpg" for n in test_gts], conf=min(cfg["detection"]["conf_grid"]), iou=op_n["nms_iou"], imgsz=640)
    lat = []
    m_n = YOLO(str(w_n))
    for p in [IMG / f"{n}.jpg" for n in list(test_gts)[:50]] * 4:
        t = time.perf_counter(); m_n.predict(str(p), conf=op_n["conf"], iou=op_n["nms_iou"], imgsz=640, verbose=False); lat.append((time.perf_counter() - t) * 1e3)
    shutil.copy2(Path(w_n).parent.parent / "results.csv", OUT / "yolo11n_results.csv")
    save("F_yolo11n_detector", {"weights": "yolo11n.pt (COCO) fine-tuned, 40 epochs, 640 px, seed 42",
        "val": gx.ultralytics_val(w_n, DET_YAML, "val", 640), "operating_point": op_n,
        "test": gx.ultralytics_val(w_n, DET_YAML, "test", 640), "test_at_operating_point": score(tp_n, test_gts, op_n["conf"]),
        "detector_latency_ms_T4": {"p50": float(np.percentile(lat[10:], 50)), "p95": float(np.percentile(lat[10:], 95))}})


In [ ]:
# G. Log the new results to W&B (aggregates only; no images)
if LOG_TO_WANDB:
    import wandb
    try:
        from google.colab import userdata; key = userdata.get("WANDB_API_KEY")
    except Exception:
        from getpass import getpass; key = getpass("W&B API key (hidden): ")
    wandb.login(key=key, relogin=True, verify=True); del key
    with wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f"phase2-gap-experiments-{STAMP}", job_type="evaluation",
                    tags=["phase2", "validation-selection", "baselines"], config={"source_commit": SHA, "weights_sha256": PRE["weights_sha256"]}) as run:
        art = wandb.Artifact("phase2-gap-experiments", type="evaluation", description="Baselines, validation sweeps, per-stage timing; aggregates only.")
        art.add_file(str(OUT / "results.json")); run.log_artifact(art); RESULTS["wandb_run"] = run.url
    save_json(RESULTS, OUT / "results.json"); print("W&B run:", RESULTS["wandb_run"])
print("Done. Results:", OUT / "results.json")
